In [ ]:
from pathlib import Path
import sys
import os
import random

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "src").exists():
    repo_root = repo_root.parent

sys.path.insert(0, str(repo_root / "src"))
sys.path.insert(0, str(repo_root / "code"))

from import_libraries import *
import common_libraries.generic_library as gn_l
import common_libraries.io_library as io_l

from ml_classification.ml_classification_pipeline import (
    GeospatialClassificationPipeline,
    ClassificationPipelineConfig,
)

# Global reproducibility seed
random_seed = 42
os.environ["PYTHONHASHSEED"] = str(random_seed)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
random.seed(random_seed)
np.random.seed(random_seed)
try:
    import tensorflow as tf
    tf.random.set_seed(random_seed)
except Exception:
    print("TensorFlow not available, skipping TensorFlow seed setting.")

In [ ]:
import common_libraries.io_library as io_l

project_dir = "C:/work_dir/test_ml_demo_parcels_local"


input_file = "C:/work_dir/satellite_monthly_sample_parcel_statistics_manager/parcels_with_statistics.gpkg"
gdf = io_l.read_data(input_file)
gdf = gdf.rename(columns={"parcel_code": "parcel_id"})
gdf.head()

In [ ]:
NON_FEATURE_COLUMNS = {
    "parcel_id", "parcel_code", "class", "label", "label_code", "initial_label", "initial_label_code", "period_start", "period_end",
    "predicted_label", "predicted_label_code", "rfv_true", "svm_novelty_full",
    "parcel_id", "period_start", "period_end", "geometry", "batch_number", "prediction_probability",
    "categ_code", "data_clean_mid", "has_rfv", "normal", "svm_outlier_full", "holdout_mid", "holdout_full", "meets_minimum_pixel_count"
}
numeric_or_boolean = set(gdf.select_dtypes(include=["number", "bool"]).columns)
features = [col for col in gdf.columns if col in numeric_or_boolean and col not in NON_FEATURE_COLUMNS]
features

In [ ]:
target_column = "class_ml"
id_column = "parcel_id"

label_counts = gdf[target_column].value_counts()
rare_labels = label_counts[label_counts < 20].index

gdf.loc[gdf[target_column].isin(rare_labels), target_column] = 9999
print(gdf[target_column].value_counts())

In [ ]:
config = ClassificationPipelineConfig(
    project_dir=project_dir,
    target_column=target_column,
    feature_columns=features,
    id_column=id_column,
    
    time_column="period_start",
    reshape_time_series=True,
    prediction_cutoff="2024-09-30",
        
    cv_folds=3,
    test_size=0.3,
    
    spatial_split=True,
    spatial_split_grid_size=10,
    spatial_split_method="by_row",  # or "by_group"
    
    random_state=random_seed,
    n_jobs=1,
    max_search_candidates=12,
    interpretability_top_models=3,
    interpretability_include_shap=False,    
    top_voting_models=None,
    
    scoring_primary="f1_macro",
    selection_type="soft_voting",
    prediction_confidence_threshold=0.60,
        
    # label_balancing_method="random_oversample",
    selected_models=(
        "gradient_boosting",
        "hist_gradient_boosting",
        "bagging",
        "random_forest",
        "decision_tree",
        "extra_trees",
        "bayesian",
        "xgboost",
        #"lightgbm",
        "knn",
        "neural_network"
   ),
    max_map_geometries=20000,
    open_html_report=True,
)
pipeline = GeospatialClassificationPipeline(config)

In [ ]:
summary_check = pipeline.run_check(gdf)
summary_check

In [ ]:
summary_prepare = pipeline.run_prepare()

In [ ]:
summary_train = pipeline.run_train()
summary_train

In [ ]:
summary_evaluate = pipeline.run_evaluate()
summary_evaluate

In [ ]:
summary_predict = pipeline.run_predict()
summary_predict

In [11]:
# Show only rows that were unknown during training
classified_gdf = pd.read_csv(os.path.join(project_dir, "05_predict", "final_predictions_preview.csv"))
classified_gdf[id_column] = classified_gdf[id_column].astype(str)
display(classified_gdf.head())

,parcel_id,class_ml,class_ml_prediction,prediction_confidence,prediction_needs_review,class_ml_filled
0,101564154,wheat,wheat,0.8765173470,False,wheat
1,101564161,stone_fruit_trees,stone_fruit_trees,0.9809594093,False,stone_fruit_trees
2,101564165,maize,maize,0.9598504495,False,maize
3,101564167,stone_fruit_trees,stone_fruit_trees,0.9896522498,False,stone_fruit_trees
4,103277508,stone_fruit_trees,stone_fruit_trees,0.9487739609,False,stone_fruit_trees


In [ ]:
classified_gdf[f"accuracy"] = classified_gdf["class_ml"] == classified_gdf["class_ml_prediction"]
accuracy = classified_gdf[f"accuracy"].mean()
print(f"Accuracy on data: {accuracy:.2%}")